Complete Preprocessing Pipeline with ColumnTransformer


Production-ready preprocessing pipeline handling mixed data types with automatic leakage prevention


In [27]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import RobustScaler
import kagglehub
from kagglehub import KaggleDatasetAdapter, dataset_load # Import dataset_load


# FIX: Specify the correct file name for the dataset
file_path = "WA_Fn-UseC_-Telco-Customer-Churn.csv"

# Load the latest version using dataset_load to avoid deprecation warning
df = dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "blastchar/telco-customer-churn",
  file_path,
)

# FIX: Print data loaded message
print(f"Data Loaded from Kaggle dataset 'blastchar/telco-customer-churn' file: {file_path}")

# FIX: Handle 'TotalCharges' column which might be object type due to spaces/empty strings
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Uncomment for data inspection
df.describe()

# FIX: Update target column for the new dataset
target = 'Churn'
y = df[target]

# 'X' is everything ELSE (drop the target column)
# We also drop IDs and dates for now, as standard models can't process them directly
# FIX: Update columns to drop for the new dataset
columns_to_drop = [target,'customerID']
X = df.drop(columns=columns_to_drop, errors='ignore')


# ============================================================
# STEP 1: Identify column types
# ============================================================
# In practice, inspect with df.info() and df.dtypes

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(include=['object','category']).columns.tolist()
# ------------------------------------------------------------

# ============================================================
# STEP 2: Build sub-pipelines for each type
# ============================================================

# impute missings with median
# numeric_pipeline = Pipeline([
#     ('imputer', SimpleImputer(strategy='median')),
#     ('scaler', StandardScaler())
# ])
# we have outliers so we should use RobustScaler()

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

#impute missing with 'Unknown', then one-hot encode
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant',fill_value='Unknown')),
    ('scaler', OneHotEncoder(handle_unknown='ignore',sparse_output=False))
])

# ============================================================
# STEP 3: Combine with ColumnTransformer
# ============================================================
preprocessor = ColumnTransformer(
    transformers = [
        ('num', numeric_pipeline, numeric_features),
        ('cat', categorical_pipeline, categorical_features)
    ],
    remainder = 'drop' # drop unlisted columns
)

# ============================================================
# STEP 4: Full pipeline = preprocessing + model
# ============================================================

full_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

# ============================================================
# STEP 5: Split FIRST, then fit (leakage-free)
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
full_pipeline.fit(X_train,y_train) # transformation + model
score = full_pipeline.score(X_test,y_test) # transformation + prediction

# Cross-validation is also leak-free with Pipeline:
scores = cross_val_score(full_pipeline, X, y, cv=5)
print(f"CV Accuracy: {scores.mean():.3f} +/- {scores.std():.3f}")

print("Pipeline steps: ", [name for name, _ in full_pipeline.steps])
print("Preprocessor transformers: ", [name for name,_,_  in preprocessor.transformers])

Using Colab cache for faster access to the 'telco-customer-churn' dataset.
Data Loaded from Kaggle dataset 'blastchar/telco-customer-churn' file: WA_Fn-UseC_-Telco-Customer-Churn.csv
CV Accuracy: 0.747 +/- 0.006
Pipeline steps:  ['preprocess', 'model']
Preprocessor transformers:  ['num', 'cat']


In [28]:
from sklearn.metrics import classification_report, confusion_matrix

# Predict on the test set
y_pred = full_pipeline.predict(X_test)

print("\n" + "=" * 60)
print("THE REALITY OF ACCURACY")
print("=" * 60)

print("\nConfusion Matrix:")
# This shows [True Negatives, False Positives]
#            [False Negatives, True Positives]
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
# This shows Precision, Recall, and F1-Score
print(classification_report(y_test, y_pred, zero_division=0))


THE REALITY OF ACCURACY

Confusion Matrix:
[[753 279]
 [ 85 292]]

Classification Report:
              precision    recall  f1-score   support

          No       0.90      0.73      0.81      1032
         Yes       0.51      0.77      0.62       377

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.79      0.74      0.75      1409

